# Objective
# Our core research question:
    Can we accurately predict apartment prices in Buenos Aires based on characteristics like size, type, and location?
    Answering that question is a three-stage process, all of which this lesson covers:
    Clean, structured data — raw data is rarely machine-learning-ready straight from the source.
    Thoughtful feature engineering — transforming raw columns into mathematical inputs a model can actually use.
    A proper train-test split — ensuring our model is evaluated on data it has never seen during training.

# Project workflow
    Used glob to locate multiple CSV files matching a naming pattern and load them into a single DataFrame
    Filtered data to a homogeneous market segment using .loc[] with lambda functions and .query()
    Removed outliers systematically using quantile-based bounds that adapt to any data distribution
    Extracted numeric and categorical features from raw string columns
    Diagnosed missing-value patterns using bar charts and the missingno matrix
    Detected and resolve multicollinearity between numeric features using a correlation heatmap
    Understood why one-hot encoding is needed for categorical variables, why category_encoders is preferred over scikit-learn's OneHotEncoder, and how to apply it
    Split data into training and test sets with scikit-learn's train_test_split

In [1]:
import numpy as np
import pandas as pd
from glob import glob
import matplotlib.pyplot as plt
import seaborn as sns

# Import and Merge Multiple CSV Files

In [2]:
datasets=glob("buenos-aires-real-estate-*.csv")
datasets

['buenos-aires-real-estate-1.csv',
 'buenos-aires-real-estate-2.csv',
 'buenos-aires-real-estate-3.csv',
 'buenos-aires-real-estate-4.csv',
 'buenos-aires-real-estate-5.csv']

In [3]:
files=[pd.read_csv(x) for x in datasets]

In [4]:
df=pd.concat(files, ignore_index=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 43029 entries, 0 to 43028
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   operation                   43029 non-null  str    
 1   property_type               43029 non-null  str    
 2   place_with_parent_names     43029 non-null  str    
 3   lat-lon                     34734 non-null  str    
 4   price                       38073 non-null  float64
 5   currency                    38072 non-null  str    
 6   price_aprox_local_currency  38073 non-null  float64
 7   price_aprox_usd             38073 non-null  float64
 8   surface_total_in_m2         29871 non-null  float64
 9   surface_covered_in_m2       36420 non-null  float64
 10  price_usd_per_m2            24449 non-null  float64
 11  price_per_m2                32642 non-null  float64
 12  floor                       6505 non-null   float64
 13  rooms                       23806 non-null

### We will wrap this into a function called merged_files so that any future notebook can call it with a single line and get back the fully merged DataFrame.

In [5]:
# def merged_files(data):
#     return pd.concat([pd.read_csv(file) for file in glob(data)], ignore_index=True)
# merged_files("buenos-aires-real-estate-*.csv").info()

# Data Cleaning

#                       Problems                and                           Effect on model if ignored
    Market heterogeneity — listings span multiple property types and cities -	Model learns inconsistent patterns; predictions are mediocre everywhere
    Extreme outliers — unusual surface area values	-Regression coefficients bend toward extreme cases, reducing accuracy for typical properties
    Missing values — some columns are >50% empty	-Filling with mean/median invents data; better to drop and avoid noise
    Redundant features — highly correlated columns	-Model cannot isolate individual feature effects; coefficients become unstable
    String-encoded numbers — coordinates stored as "-34.60,-58.38"	-Cannot be used in matrix multiplication until split and cast to float
    Data leakage — price-derived columns in the feature set	  -Model "cheats" by using the answer; fails completely on new listings

## dealing with Market heterogeneity

# Filter to Capital Federal Only
    Why Filter to Capital Federal?
    The raw dataset spans Capital Federal (the autonomous city of Buenos Aires) and Greater Buenos Aires (GBA) — two fundamentally different real estate markets that should not be modeled together.    
    Capital Federal: Dense urban core where walkability, transit proximity, building amenities, and neighborhood prestige drive high prices per square meter. The key features are size and location within the city.
    Greater Buenos Aires: Diverse suburban and semi-rural zones where lot size, commute distance, and proximity to commercial centers matter more. A different set of features drives prices.
    Mixing them produces a heterogeneous dataset — a model trained on both would be mediocre at predicting in either market, because the pricing logic differs. One linear equation cannot simultaneously capture urban Palermo dynamics and suburban La Matanza dynamics.
    Additional filters:
    Apartments only — houses and commercial spaces follow different pricing mechanics (lot size, zoning, business revenue potential)
    Price < $400,000 — caps the range to typical residential buyers, excluding luxury outliers whose prices are driven by factors our features do not capture

In [9]:
def filtered_df(data):
    return (df
            .loc[lambda x: x["place_with_parent_names"].str.contains("Capital Federal")]
            .query('property_type == "apartment"')
            .query('price_aprox_usd < 400_000')
           )
print(filtered_df(df))

      operation property_type                   place_with_parent_names  \
0          sell     apartment  |Argentina|Capital Federal|Villa Crespo|   
4          sell     apartment     |Argentina|Capital Federal|Chacarita|   
9          sell     apartment    |Argentina|Capital Federal|Villa Luro|   
11         sell     apartment          |Argentina|Capital Federal|Once|   
20         sell     apartment   |Argentina|Capital Federal|San Nicolás|   
...         ...           ...                                       ...   
42990      sell     apartment       |Argentina|Capital Federal|Liniers|   
42997      sell     apartment       |Argentina|Capital Federal|Palermo|   
43001      sell     apartment     |Argentina|Capital Federal|Mataderos|   
43002      sell     apartment     |Argentina|Capital Federal|Mataderos|   
43021      sell     apartment     |Argentina|Capital Federal|Monserrat|   

                             lat-lon     price currency  \
0      -34.6047834183,-58.4586812499  18